In [1]:
#Experiment zur Positionierung von Farbblöcken

# Der Hauptzweck der Farbblock-Positionierung besteht darin, die Funktion zur Erkennung von Farbblöcken weiterzuentwickeln.
# Das Prinzip besteht darin, die Entfernungs- und Positionsdaten des Farbblocks zur Kamera zu ermitteln
# und diese durch Berechnung der Koordinaten des Mittelpunkts des Farbblocks auf dem Kamerabildschirm zu bewerten,
# wodurch die Positionierung des Farbblocks realisiert wird. 
# Die Versuchsergebnisse zeigen, dass der Mittelpunkt des Farbblocks stets so ermittelt wird, dass er der Bewegung folgt.

In [2]:
import cv2 as cv
import threading
import random
from time import sleep
import ipywidgets as widgets
from IPython.display import display
#from positioning import color_follow

def bgr8_to_jpeg(value, quality=75):
    #return bytes(cv.imencode('.jpg', value)[1])
    return cv.imencode('.jpg', value)[1].tobytes()

In [3]:
def follow_function(self, img, HSV_config):
         (color_lower, color_upper) = HSV_config
         self.img = cv.resize(img, (640, 480), )
         self.img = cv.GaussianBlur(self.img, (5, 5), 0)
         hsv = cv.cvtColor(self.img, cv.COLOR_BGR2HSV)
         mask = cv.inRange(hsv, color_lower, color_upper)
         mask = cv.erode(mask, None, iterations=2)
         mask = cv.dilate(mask, None, iterations=2)
         mask = cv.GaussianBlur(mask, (5, 5), 0)
         cnts = cv.findContours(mask.copy(), cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)[-2]
        
         if len(cnts) > 0:
             cnt = max(cnts, key=cv.contourArea)
             (color_x, color_y), color_radius = cv.minEnclosingCircle(cnt)
            
             if color_radius > 10:
                 # Mark the detected color with the prototype coil
                 # Mark the detected color with a prototype coil
                 cv.circle(self.img, (int(color_x), int(color_y)), int(color_radius), (255, 0, 255), 3)
                 print(color_x,color_y)
         return self.img

In [4]:
import numpy as np

def follow_color_(img, lower, upper):
    color_lower = lower
    color_upper = upper

    img = cv.resize(img, (640, 480), )
    img = cv.GaussianBlur(img, (5, 5), 0)
    hsv = cv.cvtColor(img, cv.COLOR_BGR2HSV)
    mask = cv.inRange(hsv, color_lower, color_upper)
    mask = cv.erode(mask, None, iterations=2)
    mask = cv.dilate(mask, None, iterations=2)
    mask = cv.GaussianBlur(mask, (5, 5), 0)
    cnts = cv.findContours(mask.copy(), cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)[-2]

    if len(cnts) > 0:
        cnt = max(cnts, key=cv.contourArea)
        (color_x, color_y), color_radius = cv.minEnclosingCircle(cnt)

        cv.circle(img, (int(color_x), int(color_y)), int(color_radius), (255, 0, 255), 2)
        # print(color_x,color_y)
        
    return img

In [5]:
button_layout = widgets.Layout(width='200px', height='100px', align_self='center')
# Output widget Output widget
output = widgets.Output()

# Color tracking Color tracking
color_follow = widgets.Button(description='color_follow', button_style='success', layout=button_layout)

# Select color Select color
# widgets.ToggleButtons(...): Erzeugt eine Reihe von nebeneinanderliegenden Buttons. Es kann immer nur einer gleichzeitig ausgewählt sein (ähnlich wie bei Radio-Buttons).
# options=['red', 'green', 'blue', 'yellow']: Definiert die Beschriftung der Buttons und die Werte, die du auswählen kannst [2, 5].
# button_style='success': Legt das Farbschema fest. 'success' färbt den aktuell ausgewählten Button in der Regel grün ein [1, 4]. Andere Optionen wären z. B. 'info' (blau) oder 'danger' (rot).
# tooltips=[...]: Bestimmt den Text, der erscheint, wenn man mit der Maus über einen Button fährt [2, 5]. (Hinweis: In deinem Code-Schnipsel passen die Tooltip-Texte inhaltlich nicht ganz zu den Farben, da sie von "slow" und "regular" sprechen).
# choose_color: Das ist der Name der Variable, in der das Widget gespeichert wird. Über choose_color.value kannst du später im Code abfragen, welche Farbe der Nutzer angeklickt hat.
choose_color = widgets.ToggleButtons(
    options=['red', 'green', 'blue', 'yellow'], 
    button_style='success', 
    tooltips=['Description of slow', 'Description of regular', 'Description of fast'])

# Cancel tracking Cancel tracking
follow_cancel = widgets.Button(description='follow_cancel', button_style='danger', layout=button_layout)

# exit exit
exit_button = widgets.Button(description='Exit', button_style='danger', layout=button_layout)

# Image widget Image widget
imgbox = widgets.Image(format='jpg', height=480, width=640, layout=widgets.Layout(align_self='auto'))

# Vertical layout Vertical layout
img_box = widgets.VBox([imgbox, choose_color], layout=widgets.Layout(align_self='auto'))

# Vertical layout Vertical layout
Slider_box = widgets.VBox([color_follow,follow_cancel,exit_button], layout=widgets.Layout(align_self='auto'))

# Horizontal layout Horizontal layout
controls_box = widgets.HBox([img_box, Slider_box], layout=widgets.Layout(align_self='auto'))
# ['auto', 'flex-start', 'flex-end', 'center', 'baseline', 'stretch', 'inherit', 'initial', 'unset']

display(controls_box)


def on_color_change(change):
    color = change['new']
    print(f"Du hast die Farbe {change['new']} ausgewählt!")

choose_color.observe(on_color_change, names='value')


def on_exit_button_clicked(a):
    print("Exit-Button clicked")

exit_button.on_click(on_exit_button_clicked)

In [6]:
def camera():    
    global HSV_learning, model
    # Open camera Open camera
    capture = cv2.VideoCapture(0)
    capture.set(3, 640)
    capture.set(4, 480)
    capture.set(5, 30)
    # Be executed in loop when the camera is opened normally
    # Loop execution when the camera is turned on normally
    while capture.isOpened():
        try:
            _, img = capture.read()
            img = cv2.resize(img, (640, 480))

            if model == 'color_follow':
                img = follow.follow_function(img, color_hsv[choose_color.value])
                cv.putText(img, choose_color.value, (int(img.shape[0] / 2), 50), cv.FONT_HERSHEY_SIMPLEX, 2, color[random.randint(0, 254)], 2)
           
            if model == 'learning_color':
                img, HSV_learning = follow.get_hsv(img)
                
            if model == 'Exit':
                cv.destroyAllWindows()
                capture.release()
                break
                
            imgbox.value = cv.imencode('.jpg', img)[1].tobytes()
        except KeyboardInterrupt:capture.release()

In [7]:
global color, model

cap = cv.VideoCapture(0)
#cap.set(3, 640)
#cap.set(4, 480)
#cap.set(5, 30)

# model = 'color_follow'

#Initialize HSV value
color_hsv = {
    "red" : ((0, 143, 163), (11, 255, 255)),
    "green" : ((55, 80, 66), (78, 255, 255)),
    "blue" : ((110, 100, 121), (117, 255, 255)),
    "yellow": ((26, 100, 91), (32, 255, 255))
}

while cap.isOpened():
    ret, frame = cap.read()    # Liest den nächsten Frame.
    frame = cv.resize(frame, (640, 480))

    # ret: Ein Boolean (True/False), der sagt, ob das Bild erfolgreich gelesen wurde.
    # frame: Das eigentliche Bild als NumPy-Array (Pixel-Daten).
    # Prüfung: Mit cap.isOpened() checkst du, ob die Kamera/Datei erfolgreich geladen.
    
    if not ret: 
        print("Kamera konnte nicht gelesen werden")
        break

    #color_lower = np.array([0,143,163])   #rot
    #color_upper = np.array([11,255,255])
    #color_lower = np.array([55,80,66])   #green
    #color_upper = np.array([78,255,255])
    #color_lower = np.array([78,43,46])   #blue
    #color_upper = np.array([115,255,255])
    #color_lower = np.array([26,100,91])   #yellow
    #color_upper = np.array([32,255,255])

    if choose_color.value == 'red':
        frame = follow_color_(frame, np.array([0,143,163]), np.array([11,255,255]))

    if choose_color.value == 'green':
        frame = follow_color_(frame, np.array([55,80,66]), np.array([78,255,255]))

    if choose_color.value == 'blue':
        frame = follow_color_(frame, np.array([78,43,46]), np.array([115,255,255]))

    if choose_color.value == 'yellow':
        frame = follow_color_(frame, np.array([26,100,91]), np.array([32,255,255]))
        
   
    # cv.putText(img, choose_color.value, (int(img.shape[0] / 2), 50), cv.FONT_HERSHEY_SIMPLEX, 2, color[random.randint(0, 254)], 2)

    
    font = cv.FONT_HERSHEY_SIMPLEX
    # cv2.putText(img, str, origin, font, size, color, thickness)
    cv.putText(frame, 'eMurg', (40,80), font, 1, (0,255,50), 2, cv.LINE_AA)
    #cv.putText(frame, choose_color.value, (int(frame.shape[0] / 2), 50), font, 2, color[random.randint(0, 254)], 2)
    cv.putText(frame, choose_color.value, (int(frame.shape[0] / 2), 450), font, 0.8, (255,255,255), 2, cv.LINE_AA)
    #print(choose_color.value)

    # Bild anzeigen
    # cv.imshow("Webcam", frame)
    imgbox.value = cv.imencode('.jpg', frame)[1].tobytes()
    
    # Programm beenden wenn "q" gedrückt wird
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

# Kamera freigeben
cap.release()
cv.destroyAllWindows()

KeyboardInterrupt: 